# Isolation Forest — Detecção de Anomalias Térmicas (Frigório)

Pipeline de treinamento e exportação do modelo de detecção de anomalias térmicas
consumido pelo serviço de monitoramento (`ml/`).

Etapas: carregamento e limpeza dos dados → seleção do período de treino
(operação normal, porta fechada) → engenharia de atributos → treinamento do
Isolation Forest → validação no conjunto de teste → exportação do artefato.

In [ ]:
import json
import re
from datetime import datetime
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (14, 5)

# ---------------------------------------------------------------------
# Configuração
# ---------------------------------------------------------------------
CSV_PATH = "dados_frigorio.csv"   # dataset de origem (leituras do ESP32)
MODEL_DIR = Path("./model")       # diretório de saída dos artefatos
MODEL_DIR.mkdir(parents=True, exist_ok=True)

ROLLING_MEAN_WINDOW = 24          # leituras (~2 min a 5 s/leitura)
ROLLING_STD_WINDOW = 60           # leituras (~5 min a 5 s/leitura)
CONTAMINATION = 0.06
N_ESTIMATORS = 200
RANDOM_STATE = 42
PERSISTENCE_N_VALUES = [6, 12, 24]

FEATURES = [
    "Temp_DS18B20(C)",
    "media_movel_2min",
    "std_local_5min",
    "taxa_variacao_min",
    "porta_aberta_fill",
]

## 1. Carregamento e limpeza dos dados

Lê o CSV exportado pelo dispositivo, reconstrói os timestamps das leituras
registradas offline e normaliza a coluna de estado da porta.

In [ ]:
df = pd.read_csv(
    CSV_PATH,
    na_values=["ERR", "err", "Err"],
    dtype={"Data/Hora": str},
)

# Reconstrói timestamps das leituras marcadas como "Offline_Ns"
# (o dispositivo continua amostrando a cada 5 s mesmo sem conexão)
datas = []
ultima_data = None
for valor in df["Data/Hora"]:
    if isinstance(valor, str) and re.fullmatch(r"Offline_\d+s", valor):
        if ultima_data is not None:
            ultima_data = ultima_data + pd.Timedelta(seconds=5)
        datas.append(ultima_data)
    else:
        ultima_data = pd.to_datetime(valor, errors="coerce")
        datas.append(ultima_data)

df["Data/Hora"] = datas
df = df.dropna(subset=["Data/Hora"]).sort_values("Data/Hora").reset_index(drop=True)

# Normaliza a coluna de estado da porta (nem sempre presente no firmware)
if "porta_aberta" in df.columns:
    df["porta_aberta"] = pd.to_numeric(df["porta_aberta"], errors="coerce")
    porta_disponivel_desde = (
        df.loc[df["porta_aberta"].notna(), "Data/Hora"].min()
        if df["porta_aberta"].notna().any() else None
    )
else:
    df["porta_aberta"] = np.nan
    porta_disponivel_desde = None

print(f"Leituras: {len(df)}")
print(f"Período: {df['Data/Hora'].min()} a {df['Data/Hora'].max()}")
if porta_disponivel_desde is not None:
    print(f"Dado de porta disponível a partir de: {porta_disponivel_desde}")

## 2. Seleção do período de treino

O modelo deve ser treinado apenas com dados de operação normal (regime de
refrigeração estável, porta fechada). A célula abaixo localiza o fim do platô
de temperatura ambiente e sugere janelas estáveis para uso como treino.

In [ ]:
def detectar_janela_estavel(df, col="Temp_DS18B20(C)", time_col="Data/Hora",
                             janela_suavizacao=12, queda_min=5,
                             min_minutos_sustentados=5, janela_min=40,
                             passo_min=5, limiar_std=0.1):
    """Localiza o início da refrigeração real e sugere janelas estáveis para treino."""
    temp_suave = df[col].rolling(window=janela_suavizacao, min_periods=1).median()
    plato = temp_suave.round(0).mode().iloc[0]
    limiar_queda = plato - queda_min
    abaixo = temp_suave < limiar_queda

    min_leituras = int(min_minutos_sustentados * 60 / 5)
    inicio_refrigeracao, contador = None, 0
    for i in range(len(abaixo)):
        if abaixo.iloc[i]:
            contador += 1
            if contador >= min_leituras and inicio_refrigeracao is None:
                inicio_refrigeracao = df[time_col].iloc[i - min_leituras + 1]
        else:
            contador = 0

    candidatos = []
    if inicio_refrigeracao is not None:
        df_pos = df[df[time_col] >= inicio_refrigeracao].reset_index(drop=True)
        t, t_max = df_pos[time_col].min(), df_pos[time_col].max()
        while t + pd.Timedelta(minutes=janela_min) <= t_max:
            janela = df_pos[(df_pos[time_col] >= t) & (df_pos[time_col] < t + pd.Timedelta(minutes=janela_min))]
            if len(janela) > 0 and janela[col].std() <= limiar_std:
                candidatos.append((t, t + pd.Timedelta(minutes=janela_min), janela[col].std()))
            t += pd.Timedelta(minutes=passo_min)

    return plato, inicio_refrigeracao, candidatos


plato, inicio_refrigeracao, candidatos = detectar_janela_estavel(df)

print(f"Platô ambiente: {plato:.1f} °C")
print(f"Início da refrigeração: {inicio_refrigeracao}")
print(f"Janelas estáveis candidatas: {len(candidatos)}")
for inicio, fim, std_j in candidatos[:5]:
    print(f"  {inicio} -> {fim}  (std={std_j:.3f} °C)")

fig, ax = plt.subplots()
ax.plot(df["Data/Hora"], df["Temp_DS18B20(C)"], linewidth=0.8)
if inicio_refrigeracao is not None:
    ax.axvline(inicio_refrigeracao, color="tab:red", linestyle="--", label="Início da refrigeração")
ax.set_title("Temperatura ao longo do tempo")
ax.set_ylabel("°C")
ax.legend()
plt.show()

## 3. Definição do período de treino/teste

Ajustar `TRAIN_START` e `TRAIN_END` com base nos candidatos da etapa anterior.

In [ ]:
TRAIN_START = pd.Timestamp("2026-07-09 17:55:56")
TRAIN_END = pd.Timestamp("2026-07-09 21:10:56")

df_train = df[(df["Data/Hora"] >= TRAIN_START) & (df["Data/Hora"] <= TRAIN_END)].reset_index(drop=True)
df_test = df[df["Data/Hora"] > TRAIN_END].reset_index(drop=True)

print(f"Treino: {len(df_train)} leituras ({TRAIN_START} -> {TRAIN_END})")
print(f"Teste:  {len(df_test)} leituras ({TRAIN_END} -> {df['Data/Hora'].max()})")

## 4. Engenharia de atributos

Atributos: temperatura, média móvel, desvio padrão local, taxa de variação e
estado da porta. No treino, a porta é considerada sempre fechada (protocolo
de coleta).

In [ ]:
def construir_features(df, porta_col="porta_aberta"):
    df = df.copy()
    df["media_movel_2min"] = df["Temp_DS18B20(C)"].rolling(window=ROLLING_MEAN_WINDOW, min_periods=1).mean()
    df["std_local_5min"] = df["Temp_DS18B20(C)"].rolling(window=ROLLING_STD_WINDOW, min_periods=1).std().fillna(0)

    delta_t_min = df["Data/Hora"].diff().dt.total_seconds() / 60.0
    delta_temp = df["Temp_DS18B20(C)"].diff()
    df["taxa_variacao_min"] = (delta_temp / delta_t_min).replace([np.inf, -np.inf], 0).fillna(0)

    if porta_col is None:
        df["porta_aberta_fill"] = 0
    else:
        df["porta_aberta_fill"] = df[porta_col].fillna(0).astype(int)

    return df


df_train_feat = construir_features(df_train, porta_col=None)
df_test_feat = construir_features(df_test, porta_col="porta_aberta")

## 5. Treinamento do Isolation Forest

In [ ]:
def treinar_isolation_forest(df_treino, features, contamination=CONTAMINATION,
                              n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE):
    X = df_treino[features].values
    modelo = IsolationForest(contamination=contamination, n_estimators=n_estimators, random_state=random_state)
    modelo.fit(X)
    return modelo


modelo = treinar_isolation_forest(df_train_feat, FEATURES)

## 6. Validação no conjunto de teste

Aplica o modelo e o filtro de persistência (segundo estágio do sistema de
alerta) como checagem de sanidade antes da exportação.

In [ ]:
def aplicar_modelo(modelo, df_feat, features):
    df_feat = df_feat.copy()
    df_feat["score_anomalia"] = modelo.decision_function(df_feat[features].values) * -1
    df_feat["anomalia"] = (modelo.predict(df_feat[features].values) == -1).astype(int)
    return df_feat


def filtro_persistencia(flags_anomalia, n):
    flags = np.asarray(flags_anomalia)
    alerta = np.zeros(len(flags), dtype=bool)
    contador = 0
    for i, f in enumerate(flags):
        contador = contador + 1 if f == 1 else 0
        alerta[i] = contador >= n
    return alerta


resultado_teste = aplicar_modelo(modelo, df_test_feat, FEATURES)

for n in PERSISTENCE_N_VALUES:
    resultado_teste[f"alerta_N{n}"] = filtro_persistencia(resultado_teste["anomalia"], n)

print(f"Leituras de teste: {len(resultado_teste)}")
print(f"Flags de anomalia (score bruto): {resultado_teste['anomalia'].sum()} "
      f"({resultado_teste['anomalia'].mean() * 100:.1f}%)")
for n in PERSISTENCE_N_VALUES:
    print(f"Alertas persistentes (N={n}): {resultado_teste[f'alerta_N{n}'].sum()}")

## 7. Exportação do modelo

Gera `isolation_forest.joblib` (modelo) e `isolation_forest_metadata.json`
(parâmetros e período de treino) para uso pelo serviço em `ml/`.

In [ ]:
metadata = {
    "features": FEATURES,
    "contamination": CONTAMINATION,
    "n_estimators": N_ESTIMATORS,
    "random_state": RANDOM_STATE,
    "rolling_mean_window": ROLLING_MEAN_WINDOW,
    "rolling_std_window": ROLLING_STD_WINDOW,
    "persistence_n_values": PERSISTENCE_N_VALUES,
    "train_start": str(TRAIN_START),
    "train_end": str(TRAIN_END),
    "exported_at": datetime.now().isoformat(),
}

joblib.dump(modelo, MODEL_DIR / "isolation_forest.joblib")
with open(MODEL_DIR / "isolation_forest_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"Modelo exportado em: {MODEL_DIR / 'isolation_forest.joblib'}")
print(f"Metadados exportados em: {MODEL_DIR / 'isolation_forest_metadata.json'}")